# 🔍 Crypto Syndicate Research Notebook

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
from dotenv import load_dotenv
load_dotenv(os.path.abspath('../../.env'))
from crypto_syndicate.discovery import DiscoveryPipeline
from crypto_syndicate.graph import SyndicateGraph
from crypto_syndicate.report import generate_report
print("Setup complete")

## Step 1: Discover Suspicious Wallets

In [ ]:
pipeline = DiscoveryPipeline(mock_mode=True)
wallets = pipeline.run_pipeline(chains=["sol"])
print(f"Discovered {len(wallets)} wallet records")
for w in sorted(wallets, key=lambda x: x.get('suspicion_score',0), reverse=True)[:5]:
    print(f"  {w['wallet_address'][:16]}... score={w['suspicion_score']:.1f} patterns={w['patterns_flagged']}")

## Step 2: Build Graph & Detect Syndicates

In [ ]:
sg = SyndicateGraph()
sg.build_graph(wallets, pipeline.funding_relationships)
clusters = sg.detect_clusters()
print(f"Detected {len(clusters)} syndicate clusters")
for c in clusters[:5]:
    print(f"  {c.cluster_id[:24]} | score={c.suspicion_score:.1f} | wallets={len(c.wallets)} | patterns={list(c.flagged_patterns)}")

## Step 3: Interactive Network Graph

In [ ]:
from pyvis.network import Network
import json

graph_data = sg.export_graph_json(clusters)
net = Network(height="600px", width="100%", bgcolor="#1a1a24", font_color="white", notebook=True)
net.barnes_hut(gravity=-8000, central_gravity=0.3, spring_length=120)

colors = ["#4e79a7","#f28e2b","#e15759","#76b7b2","#59a14f","#edc948","#b07aa1","#ff9da7","#9c755f","#bab0ac"]
cluster_color_map = {}
for i, c in enumerate(clusters):
    for w in c.wallets:
        cluster_color_map[w] = colors[i % len(colors)]

for node in graph_data["nodes"]:
    color = cluster_color_map.get(node["id"], "#888888")
    size = max(10, node["score"] / 5 + 5)
    title = f"Wallet: {node['id'][:20]}...<br>Score: {node['score']:.1f}<br>Patterns: {', '.join(node['patterns'])}<br>Chain: {node['chain']}"
    net.add_node(node["id"], label=node["id"][:8]+"...", color=color, size=size, title=title)

for link in graph_data["links"]:
    color = "#f0a050" if link["type"] == "funding" else "#5580aa"
    net.add_edge(link["source"], link["target"], color=color, width=link.get("weight",1))

net.show("syndicate_graph.html")
print(f"Graph saved: {len(graph_data['nodes'])} nodes, {len(graph_data['links'])} edges")

## Step 4: Token Launch Timeline

In [ ]:
import plotly.graph_objects as go
import pandas as pd

rows = []
for w in wallets:
    for token in w.get("tokens_traded", []):
        rows.append({"wallet": w["wallet_address"][:12]+"...", "token": token[:12]+"...",
                     "score": w.get("suspicion_score",0), "patterns": "|".join(w.get("patterns_flagged",[]))})
if rows:
    df = pd.DataFrame(rows)
    fig = go.Figure(go.Scatter(
        x=df["token"], y=df["wallet"], mode="markers",
        marker=dict(size=df["score"]/3+5, color=df["score"], colorscale="Reds",
                    showscale=True, colorbar=dict(title="Suspicion Score")),
        text=df["patterns"], hovertemplate="<b>%{y}</b><br>Token: %{x}<br>Score: %{marker.color:.1f}<br>Patterns: %{text}<extra></extra>"
    ))
    fig.update_layout(title="Wallet × Token Activity Map", plot_bgcolor="#1a1a24",
                      paper_bgcolor="#0f0f14", font_color="#e0e0e0", height=500)
    fig.show()
else:
    print("No timeline data available")

## Step 5: Top Syndicates Ranking

In [ ]:
import pandas as pd
if clusters:
    data = [{"Rank": i+1, "Cluster ID": c.cluster_id[:24], "Score": f"{c.suspicion_score:.1f}",
             "Wallets": len(c.wallets), "Patterns": ", ".join(c.flagged_patterns),
             "Est. Profit USD": f"${c.estimated_profit_usd:,.0f}"} for i, c in enumerate(clusters[:20])]
    df = pd.DataFrame(data)
    display(df)
else:
    print("No clusters detected")

## Step 6: Generate Full Report

In [ ]:
paths = generate_report(wallets, clusters, output_dir="../../results_notebook")
print("Reports generated:")
for k, v in paths.items():
    print(f"  {k}: {v}")

## Step 7: Start Live Monitoring (Background)

In [ ]:
import threading
from crypto_syndicate.monitor import MonitoringLoop

def start_monitor():
    loop = MonitoringLoop(chains=["sol"], poll_interval=300, output_dir="../../results_monitor")
    loop.run()

monitor_thread = threading.Thread(target=start_monitor, daemon=True)
monitor_thread.start()
print(f"Monitor started in background thread (daemon=True)")
print("Alerts will be written to: results_monitor/alerts.json and alerts.log")
print("Stop: restart the kernel to kill the daemon thread")